# 🏥 Kvasir-VQA-x1 — BLIP-2 LoRA Fine-Tuning

Fine-tune BLIP-2 (`Salesforce/blip2-opt-2.7b`) on the Kvasir-VQA-x1 medical VQA dataset using LoRA (Low-Rank Adaptation).

**Requirements:**
- GPU runtime: `Runtime > Change runtime type > T4 GPU`
- ~16GB GPU RAM (T4 is sufficient with 8-bit quantization)
- ~2-3 hours training time on 10K samples, 3 epochs

**What this notebook does:**
1. Downloads the Kvasir-VQA-x1 dataset (images + QA pairs)
2. Loads BLIP-2 in 8-bit quantization
3. Applies LoRA adapters to the OPT language model
4. Fine-tunes on a stratified 10K subset
5. Evaluates and compares against the zero-shot baseline

## Cell 1: Install Dependencies

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets Pillow tqdm pandas scikit-learn

## Cell 2: Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Modify these settings as needed
# ============================================================

# --- Data Source ---
USE_DRIVE = False  # True = load from Google Drive, False = download fresh
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/AI-ML-based-approaches-for-the-medical-sector"

# --- Model ---
MODEL_NAME = "Salesforce/blip2-opt-2.7b"

# --- Training ---
TRAIN_SUBSET_SIZE = 10000   # Stratified subset for training
EVAL_SAMPLES = 50           # Samples for evaluation
EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8        # Effective batch size = 32
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_NEW_TOKENS = 64

# --- LoRA ---
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# --- Paths ---
PROJECT_DIR = DRIVE_PROJECT_DIR if USE_DRIVE else "/content/kvasir-vqa"
DATA_DIR = f"{PROJECT_DIR}/data"
IMAGE_DIR = f"{DATA_DIR}/images"
RESULTS_DIR = f"{PROJECT_DIR}/results"
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SEED = 42
print(f"Config ready. USE_DRIVE={USE_DRIVE}")
print(f"Project dir: {PROJECT_DIR}")

## Cell 3: Mount Drive (if using) & Download Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")

# Check if data already exists
train_csv = Path(DATA_DIR) / "kvasir_vqa_x1_train.csv"
test_csv = Path(DATA_DIR) / "kvasir_vqa_x1_test.csv"

if train_csv.exists() and test_csv.exists():
    print(f"Data already exists at {DATA_DIR}")
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(test_csv)
else:
    print("Downloading dataset from HuggingFace...")
    from datasets import load_dataset

    # Download QA pairs
    ds = load_dataset("SimulaMet/Kvasir-VQA-x1")
    train_df = ds['train'].to_pandas()
    test_df = ds['test'].to_pandas()
    train_df.to_csv(train_csv, index=False)
    test_df.to_csv(test_csv, index=False)

    # Download images
    print("Downloading images...")
    img_ds = load_dataset("SimulaMet-HOST/Kvasir-VQA", split="raw")
    from tqdm.auto import tqdm
    for sample in tqdm(img_ds, desc="Saving images"):
        img_id = sample.get('img_id', sample.get('imgId', ''))
        img = sample.get('img', sample.get('image', None))
        if img is not None and img_id:
            save_path = Path(IMAGE_DIR) / f"{img_id}.jpg"
            if not save_path.exists():
                img.save(str(save_path))

print(f"Train: {len(train_df):,} QA pairs")
print(f"Test:  {len(test_df):,} QA pairs")
print(f"Images: {len(list(Path(IMAGE_DIR).glob('*.jpg'))):,}")

## Cell 4: Create Stratified Training Subset

In [ ]:
import re
from collections import Counter

# ---- Utility functions ----
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

def compute_word_f1(prediction, ground_truth):
    pred_tokens = normalize_text(prediction).split()
    gt_tokens = normalize_text(ground_truth).split()
    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0
    common = sum((Counter(pred_tokens) & Counter(gt_tokens)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

def compute_exact_match(prediction, ground_truth):
    return normalize_text(prediction) == normalize_text(ground_truth)

# ---- Create stratified subset ----
np.random.seed(SEED)

# Ensure complexity column exists
if 'complexity' not in train_df.columns:
    print("WARNING: 'complexity' column not found, using random sampling")
    train_subset = train_df.sample(n=min(TRAIN_SUBSET_SIZE, len(train_df)), random_state=SEED)
else:
    levels = sorted(train_df['complexity'].unique())
    per_level = TRAIN_SUBSET_SIZE // len(levels)
    subsets = []
    for level in levels:
        level_df = train_df[train_df['complexity'] == level]
        n = min(per_level, len(level_df))
        subsets.append(level_df.sample(n=n, random_state=SEED))
    train_subset = pd.concat(subsets, ignore_index=True)
    train_subset = train_subset.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Validation subset from test data
eval_subset = test_df.sample(n=min(EVAL_SAMPLES, len(test_df)), random_state=SEED)

print(f"\nTraining subset: {len(train_subset):,} samples")
if 'complexity' in train_subset.columns:
    for level in sorted(train_subset['complexity'].unique()):
        n = len(train_subset[train_subset['complexity'] == level])
        print(f"  Level {level}: {n:,}")
print(f"Evaluation subset: {len(eval_subset):,} samples")

## Cell 5: Load BLIP-2 with 8-bit Quantization + LoRA

In [ ]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Load processor
processor = Blip2Processor.from_pretrained(MODEL_NAME)

# Load model with 8-bit quantization
print(f"\nLoading {MODEL_NAME} in 8-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

# Apply LoRA
print("\nApplying LoRA adapters...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")
print("Model loaded and LoRA applied!")

## Cell 6: Training Loop

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import time
import json


class VQAFineTuneDataset(Dataset):
    """Dataset for BLIP-2 fine-tuning."""

    def __init__(self, df, image_dir, processor, max_length=64):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.max_length = max_length
        # Pre-filter to only rows with existing images
        valid_mask = self.df['img_id'].apply(
            lambda x: (self.image_dir / f"{x}.jpg").exists()
        )
        self.df = self.df[valid_mask].reset_index(drop=True)
        print(f"  Dataset: {len(self.df)} samples (after filtering missing images)")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.image_dir / f"{row['img_id']}.jpg"
        image = Image.open(img_path).convert('RGB')

        question = str(row['question'])
        answer = str(row['answer'])

        # Process inputs
        prompt = f"Question: {question} Answer:"
        encoding = self.processor(
            images=image,
            text=prompt,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )

        # Tokenize the target answer
        target_encoding = self.processor.tokenizer(
            answer,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )

        # Create labels (with -100 for padding)
        labels = target_encoding.input_ids.squeeze()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            'pixel_values': encoding.pixel_values.squeeze(),
            'input_ids': encoding.input_ids.squeeze(),
            'attention_mask': encoding.attention_mask.squeeze(),
            'labels': labels,
        }


# Create datasets
print("Creating training dataset...")
train_dataset = VQAFineTuneDataset(train_subset, IMAGE_DIR, processor)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

# Optimizer and scheduler
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM_STEPS
warmup_steps = int(total_steps * WARMUP_RATIO)

from transformers import get_cosine_schedule_with_warmup
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"\nTraining config:")
print(f"  Samples: {len(train_dataset):,}")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} accum = {BATCH_SIZE * GRAD_ACCUM_STEPS} effective")
print(f"  Steps/epoch: {len(train_loader)}")
print(f"  Total optimizer steps: {total_steps}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Epochs: {EPOCHS}")

# ---- Training ----
model.train()
training_log = []
best_loss = float('inf')
start_time = time.time()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    step_count = 0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(pbar):
        pixel_values = batch['pixel_values'].to(model.device, dtype=torch.float16)
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        labels = batch['labels'].to(model.device)

        outputs = model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        epoch_loss += outputs.loss.item()
        step_count += 1

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        # Update progress bar
        avg_loss = epoch_loss / step_count
        pbar.set_postfix({
            'loss': f'{avg_loss:.4f}',
            'lr': f'{scheduler.get_last_lr()[0]:.2e}'
        })

    # End of epoch
    avg_epoch_loss = epoch_loss / step_count
    elapsed = time.time() - start_time

    log_entry = {
        'epoch': epoch + 1,
        'avg_loss': avg_epoch_loss,
        'elapsed_min': elapsed / 60,
    }
    training_log.append(log_entry)

    print(f"\n  Epoch {epoch+1}: Loss={avg_epoch_loss:.4f}, Time={elapsed/60:.1f}min")

    # Save best model
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        model.save_pretrained(f"{CHECKPOINT_DIR}/best_lora")
        print(f"  ✓ Saved best checkpoint (loss={best_loss:.4f})")

# Save final model
model.save_pretrained(f"{CHECKPOINT_DIR}/final_lora")
print(f"\nTraining complete! Total time: {(time.time()-start_time)/60:.1f} min")
print(f"Best loss: {best_loss:.4f}")

# Save training log
with open(f"{RESULTS_DIR}/training_log.json", 'w') as f:
    json.dump(training_log, f, indent=2)
print(f"Training log saved to {RESULTS_DIR}/training_log.json")

## Cell 7: Evaluate Fine-Tuned Model

In [ ]:
# Switch to eval mode
model.eval()

predictions = []
ground_truths = []
complexities = []
questions_list = []

print(f"Running inference on {len(eval_subset)} test samples...\n")

for idx, (_, row) in enumerate(tqdm(eval_subset.iterrows(), total=len(eval_subset), desc="Evaluating")):
    img_path = Path(IMAGE_DIR) / f"{row['img_id']}.jpg"
    if not img_path.exists():
        continue

    image = Image.open(img_path).convert('RGB')
    question = str(row['question'])
    gt_answer = str(row['answer'])

    prompt = f"Question: {question} Answer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device, dtype=torch.float16)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    # Decode only generated tokens (skip prompt)
    prompt_len = inputs['input_ids'].shape[1]
    generated_tokens = generated[0][prompt_len:]
    prediction = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    predictions.append(prediction)
    ground_truths.append(gt_answer)
    complexities.append(int(row.get('complexity', 1)))
    questions_list.append(question)

print(f"\nEvaluated {len(predictions)} samples.")

## Cell 8: Results Summary & Comparison

In [ ]:
# ---- Compute metrics ----
exact_matches = 0
f1_scores = []
partial_matches = 0
per_complexity = {}

for pred, gt, comp in zip(predictions, ground_truths, complexities):
    em = compute_exact_match(pred, gt)
    f1 = compute_word_f1(pred, gt)
    exact_matches += int(em)
    f1_scores.append(f1)
    if f1 >= 0.5:
        partial_matches += 1

    # Per-complexity
    key = f"level_{comp}"
    if key not in per_complexity:
        per_complexity[key] = {'em': 0, 'f1': [], 'total': 0}
    per_complexity[key]['em'] += int(em)
    per_complexity[key]['f1'].append(f1)
    per_complexity[key]['total'] += 1

total = len(predictions)
avg_f1 = np.mean(f1_scores) * 100
em_acc = exact_matches / total * 100
pm_rate = partial_matches / total * 100

# ---- Print comparison ----
print("="*70)
print("  RESULTS COMPARISON: Zero-Shot Baseline vs Fine-Tuned BLIP-2")
print("="*70)
print(f"")
print(f"{'Metric':<30} {'Baseline':>15} {'Fine-Tuned':>15} {'Δ':>10}")
print(f"{'-'*70}")
print(f"{'Exact Match Accuracy':<30} {'0.0%':>15} {f'{em_acc:.1f}%':>15} {f'+{em_acc:.1f}%':>10}")
print(f"{'Average Word F1':<30} {'27.0%':>15} {f'{avg_f1:.1f}%':>15} {f'+{avg_f1-27.0:.1f}%':>10}")
print(f"{'Partial Match (F1≥0.5)':<30} {'5.6%':>15} {f'{pm_rate:.1f}%':>15} {f'+{pm_rate-5.6:.1f}%':>10}")
print(f"{'Samples Evaluated':<30} {'18':>15} {f'{total}':>15} {'':>10}")

print(f"\nPer-Complexity Breakdown (Fine-Tuned):")
baseline_f1 = {'level_1': 18.9, 'level_2': 27.0, 'level_3': 35.1}
for key in sorted(per_complexity.keys()):
    stats = per_complexity[key]
    level_f1 = np.mean(stats['f1']) * 100
    level_em = stats['em'] / stats['total'] * 100
    bl_f1 = baseline_f1.get(key, 0)
    print(f"  {key}: EM={level_em:.1f}%, F1={level_f1:.1f}% (baseline: {bl_f1:.1f}%, Δ={level_f1-bl_f1:+.1f}%) [n={stats['total']}]")

print(f"\n{'='*70}")

# ---- Save results ----
finetuned_summary = {
    'model': MODEL_NAME,
    'method': 'LoRA fine-tuned',
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'training_samples': len(train_dataset),
    'epochs': EPOCHS,
    'num_eval_samples': total,
    'exact_match_accuracy': em_acc,
    'average_word_f1': avg_f1,
    'exact_matches': exact_matches,
    'partial_matches_f1_gte_50': partial_matches,
    'per_complexity': {
        k: {
            'exact_matches': v['em'],
            'total': v['total'],
            'exact_accuracy': v['em'] / v['total'] * 100,
            'avg_word_f1': round(np.mean(v['f1']) * 100, 1),
        }
        for k, v in per_complexity.items()
    },
}

os.makedirs(f"{RESULTS_DIR}/predictions", exist_ok=True)

with open(f"{RESULTS_DIR}/predictions/finetuned_summary.json", 'w') as f:
    json.dump(finetuned_summary, f, indent=2)

# Save individual predictions
pred_df = pd.DataFrame({
    'question': questions_list,
    'ground_truth': ground_truths,
    'prediction': predictions,
    'complexity': complexities,
    'exact_match': [compute_exact_match(p, g) for p, g in zip(predictions, ground_truths)],
    'word_f1': [round(compute_word_f1(p, g), 3) for p, g in zip(predictions, ground_truths)],
})
pred_df.to_csv(f"{RESULTS_DIR}/predictions/finetuned_predictions.csv", index=False)

print(f"\nResults saved to {RESULTS_DIR}/predictions/")

## Cell 9: Sample Predictions (Visual Comparison)

In [ ]:
# Show detailed predictions
print("\nSample Predictions (Fine-Tuned BLIP-2 with LoRA)")
print("="*80)

for i in range(min(20, len(predictions))):
    f1 = compute_word_f1(predictions[i], ground_truths[i])
    em = compute_exact_match(predictions[i], ground_truths[i])

    if em:
        label = "EXACT ✓"
    elif f1 >= 0.5:
        label = "PARTIAL ~"
    else:
        label = "WRONG ✗"

    print(f"[{i+1}/{min(20, len(predictions))}] {label} (F1: {f1:.2f})")
    print(f"  Complexity: {complexities[i]}")
    print(f"  Question:   {questions_list[i][:100]}")
    print(f"  GT Answer:  {ground_truths[i][:100]}")
    print(f"  Predicted:  {predictions[i][:100]}")
    print("-"*80)

## Cell 10: Download Results

Run this cell to download all output files to your local machine.

In [ ]:
# List all output files
print("Output files:")
for root, dirs, files in os.walk(RESULTS_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        print(f"  {fpath} ({size/1024:.1f} KB)")

print(f"\nCheckpoint files:")
for root, dirs, files in os.walk(CHECKPOINT_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        print(f"  {fpath} ({size/1024:.1f} KB)")

# If not using Drive, offer download
if not USE_DRIVE:
    print("\n--- Download key files ---")
    from google.colab import files
    try:
        files.download(f"{RESULTS_DIR}/predictions/finetuned_summary.json")
        files.download(f"{RESULTS_DIR}/predictions/finetuned_predictions.csv")
        files.download(f"{RESULTS_DIR}/training_log.json")
        print("Downloads initiated!")
    except Exception as e:
        print(f"Auto-download failed: {e}")
        print("You can manually download files from the Files panel on the left.")